In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# set random seeds for reproducibility
torch.manual_seed(42)


In [ ]:
# check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [ ]:
df= pd.read_csv("/content/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.shape

(60000, 785)

In [ ]:
# train-test split
X = df.iloc[:,1:].values
y = df.iloc[:,0].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# standard scaling the features....recommended to train the neural network model
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
# create train dataset object
train_dataset = CustomDataset(X_train, y_train)

In [ ]:
# create test dataset object
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
# create train and test dataloader
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)


In [ ]:
# define NN class...a neural network
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model =  nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 10)
    )

  def forward(self, x):
    return self.model(x)

In [ ]:
# set learning rate and epoch...parameters
epochs = 100
learning_rate = 0.1

In [ ]:
# instantiate NN model
model = MyNN(X_train.shape[1])

# shift model over to GPU after model instantiation
model = model.to(device)

# Loss calculation
loss_fn = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(),lr = learning_rate, weight_decay=1e-4)


In [ ]:
# training loop
for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dataloader:

    # move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    output = model(batch_features)

    # loss calculation
    loss = loss_fn(output, batch_labels)

    # backtracking
    optimizer.zero_grad()
    loss.backward()

    # update gradients
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_epoch_loss = total_epoch_loss/len(train_dataloader)
  print(f"Epoch: {epoch+1} , Loss: {avg_epoch_loss}")

Epoch: 1 , Loss: 0.6600763026078542
Epoch: 2 , Loss: 0.4828798488775889
Epoch: 3 , Loss: 0.44511856937408445
Epoch: 4 , Loss: 0.4199611185391744
Epoch: 5 , Loss: 0.4031246593395869
Epoch: 6 , Loss: 0.38379810969034833
Epoch: 7 , Loss: 0.3737612833579381
Epoch: 8 , Loss: 0.36266634277502696
Epoch: 9 , Loss: 0.35413970029354097
Epoch: 10 , Loss: 0.34722727227211
Epoch: 11 , Loss: 0.33826314596335094
Epoch: 12 , Loss: 0.3332169318596522
Epoch: 13 , Loss: 0.3245857503414154
Epoch: 14 , Loss: 0.31754050227006275
Epoch: 15 , Loss: 0.3120912876526515
Epoch: 16 , Loss: 0.3087374883890152
Epoch: 17 , Loss: 0.30144521963596344
Epoch: 18 , Loss: 0.2996711250940959
Epoch: 19 , Loss: 0.29527214415868125
Epoch: 20 , Loss: 0.294573765317599
Epoch: 21 , Loss: 0.2901534451643626
Epoch: 22 , Loss: 0.28206028230985003
Epoch: 23 , Loss: 0.2794435977935791
Epoch: 24 , Loss: 0.28130520272254944
Epoch: 25 , Loss: 0.273934503475825
Epoch: 26 , Loss: 0.2745944299300512
Epoch: 27 , Loss: 0.26882679160435996
Epo

In [ ]:
# set model to eval mode
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
# evaluation code
total=0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in test_dataloader:
    # move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    output = model(batch_features)
    _, predicted = torch.max(output, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")


Accuracy: 0.8925


In [ ]:
# evaluation code
total=0
correct=0
with torch.no_grad():
  for batch_features, batch_labels in train_dataloader:
    # move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    output = model(batch_features)
    _, predicted = torch.max(output, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")


Accuracy: 0.9605208333333334
